# Corrosion Data Analysis — NIST CORR-DATA

**Goal:** analyze corrosion-resistance ratings (A/B/C/D) from the NIST CORR-DATA database and evaluate how material, environment, concentration, and temperature contribute to classification.

This notebook emphasizes **careful preprocessing and honest validation**. Original source values are preserved; ambiguous values are not silently converted into invented numbers.

## 1. Load the NIST data

Place the downloaded `CORR-DATA_Database.csv` in the project's `data/` folder. The official source links are listed in the project README.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/CORR-DATA_Database.csv')
print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

## 2. Clean fields and extract clear A/B/C/D ratings

Only clear qualitative ratings are retained for classification. Ambiguous entries such as `A (Resistant)/C (Questionable)` are not forced into one class.

In [ ]:
for column in ['Environment', 'Material Group', 'Material Family', 'Material']:
    df[column + '_clean'] = df[column].astype('string').str.strip()

df['material_group_clean'] = df['Material Group_clean'].replace({'MIscellaneous': 'Miscellaneous'})
rating_text = df['Rate (mm/yr) or Rating'].astype('string').str.strip()
clear_rating_mask = (
    rating_text.str.match(r'^[ABCD]\s*\([^/)]*\)\s*$', na=False)
    | rating_text.str.fullmatch(r'[ABCD]', na=False)
)
df['corrosion_rating'] = rating_text.where(clear_rating_mask).str.extract(r'^([ABCD])', expand=False)
rated_df = df[df['corrosion_rating'].notna()].copy()

print('Rated observations:', len(rated_df))
print(rated_df['corrosion_rating'].value_counts())

## 3. Baseline classification dataset

Rows with missing material names are removed because material is a core input. Missing concentration or temperature does not cause a row to be deleted.

In [ ]:
classification_df = rated_df[[
    'material_group_clean', 'Material Family_clean', 'Material_clean',
    'Environment_clean', 'corrosion_rating'
]].dropna().copy()

classification_df.to_csv('data/processed/corrosion_classification_baseline.csv', index=False)
print('Baseline dataset:', classification_df.shape)

## 4. Safe temperature processing

Single numerical temperatures are retained. Clear numerical ranges such as `0-118` are converted to a minimum, maximum, and midpoint. Qualitative text is not guessed.

In [ ]:
temperature_text = rated_df['Temperature (deg C)'].astype('string').str.strip()
rated_df['temperature_numeric'] = pd.to_numeric(temperature_text, errors='coerce')
temperature_range_mask = temperature_text.str.match(r'^\s*-?\d+(?:\.\d+)?\s*-\s*-?\d+(?:\.\d+)?\s*$', na=False)
rated_df['temperature_min_c'] = rated_df['temperature_numeric']
rated_df['temperature_max_c'] = rated_df['temperature_numeric']
parts = temperature_text[temperature_range_mask].str.split('-', expand=True)
rated_df.loc[temperature_range_mask, 'temperature_min_c'] = pd.to_numeric(parts[0], errors='coerce')
rated_df.loc[temperature_range_mask, 'temperature_max_c'] = pd.to_numeric(parts[1], errors='coerce')
rated_df['temperature_mid_c'] = (rated_df['temperature_min_c'] + rated_df['temperature_max_c']) / 2
print(rated_df['temperature_mid_c'].describe())

## 5. Safe concentration processing

Only numerical Vol-% values from 0 to 100 are treated as numerical concentration. Clear ranges are represented by their midpoint. Values above 100, including observed 117 and 672 entries, are not silently reinterpreted.

In [ ]:
concentration_text = rated_df['Concentration (Vol %)'].astype('string').str.strip()
rated_df['concentration_min'] = pd.to_numeric(concentration_text, errors='coerce')
rated_df['concentration_max'] = rated_df['concentration_min']
invalid_single = (rated_df['concentration_min'] < 0) | (rated_df['concentration_min'] > 100)
rated_df.loc[invalid_single, ['concentration_min', 'concentration_max']] = np.nan
concentration_range_mask = concentration_text.str.match(r'^\s*\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*$', na=False)
parts = concentration_text[concentration_range_mask].str.split('-', expand=True)
cmin = pd.to_numeric(parts[0], errors='coerce')
cmax = pd.to_numeric(parts[1], errors='coerce')
valid = (cmin >= 0) & (cmin <= 100) & (cmax >= 0) & (cmax <= 100)
idx = cmin[valid].index
rated_df.loc[idx, 'concentration_min'] = cmin.loc[idx]
rated_df.loc[idx, 'concentration_max'] = cmax.loc[idx]
rated_df['concentration_mid'] = (rated_df['concentration_min'] + rated_df['concentration_max']) / 2
print(rated_df['concentration_mid'].describe())

## 6. Random-split models

An 80/20 stratified split is used for a controlled comparison. Missing numerical values are filled using medians calculated from training data only, with separate missing-value indicators.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
from scipy.sparse import hstack

X = classification_df[['material_group_clean', 'Material Family_clean', 'Material_clean', 'Environment_clean']]
y = classification_df['corrosion_rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)
model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model.fit(X_train_encoded, y_train)
y_pred = model.predict(X_test_encoded)
accuracy = accuracy_score(y_test, y_pred)
print(f'Baseline accuracy: {accuracy:.2%}')
print(classification_report(y_test, y_pred))

## 7. Concentration-enhanced model

Missing concentration is filled with the training-set median and a `concentration_missing` indicator is retained.

In [ ]:
tmp = rated_df[['material_group_clean','Material Family_clean','Material_clean','Environment_clean','concentration_mid','corrosion_rating']].dropna(subset=['Material_clean']).copy()
X2 = tmp.drop(columns='corrosion_rating')
y2 = tmp['corrosion_rating']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.20, random_state=42, stratify=y2)
X2_train = X2_train.copy(); X2_test = X2_test.copy()
median_c = X2_train['concentration_mid'].median()
X2_train['concentration_missing'] = X2_train['concentration_mid'].isna().astype(int)
X2_test['concentration_missing'] = X2_test['concentration_mid'].isna().astype(int)
X2_train['concentration_mid'] = X2_train['concentration_mid'].fillna(median_c)
X2_test['concentration_mid'] = X2_test['concentration_mid'].fillna(median_c)
cat = ['material_group_clean','Material Family_clean','Material_clean','Environment_clean']
enc2 = OneHotEncoder(handle_unknown='ignore')
E2_train = enc2.fit_transform(X2_train[cat]); E2_test = enc2.transform(X2_test[cat])
F2_train = hstack([E2_train, X2_train[['concentration_mid','concentration_missing']].to_numpy(dtype=np.float64)])
F2_test = hstack([E2_test, X2_test[['concentration_mid','concentration_missing']].to_numpy(dtype=np.float64)])
model2 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(F2_train, y2_train)
pred2 = model2.predict(F2_test)
accuracy2 = accuracy_score(y2_test, pred2)
print(f'Model 2 accuracy: {accuracy2:.2%}')
print(classification_report(y2_test, pred2))

## 8. Temperature-enhanced model

Temperature is added using the same leakage-safe procedure.

In [ ]:
tmp3 = rated_df[['material_group_clean','Material Family_clean','Material_clean','Environment_clean','concentration_mid','temperature_mid_c','corrosion_rating']].dropna(subset=['Material_clean']).copy()
X3 = tmp3.drop(columns='corrosion_rating'); y3 = tmp3['corrosion_rating']
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.20, random_state=42, stratify=y3)
X3_train = X3_train.copy(); X3_test = X3_test.copy()
median_c3 = X3_train['concentration_mid'].median(); median_t3 = X3_train['temperature_mid_c'].median()
X3_train['concentration_missing'] = X3_train['concentration_mid'].isna().astype(int); X3_test['concentration_missing'] = X3_test['concentration_mid'].isna().astype(int)
X3_train['temperature_missing'] = X3_train['temperature_mid_c'].isna().astype(int); X3_test['temperature_missing'] = X3_test['temperature_mid_c'].isna().astype(int)
X3_train['concentration_mid'] = X3_train['concentration_mid'].fillna(median_c3); X3_test['concentration_mid'] = X3_test['concentration_mid'].fillna(median_c3)
X3_train['temperature_mid_c'] = X3_train['temperature_mid_c'].fillna(median_t3); X3_test['temperature_mid_c'] = X3_test['temperature_mid_c'].fillna(median_t3)
enc3 = OneHotEncoder(handle_unknown='ignore')
E3_train = enc3.fit_transform(X3_train[cat]); E3_test = enc3.transform(X3_test[cat])
num_cols3 = ['concentration_mid','concentration_missing','temperature_mid_c','temperature_missing']
F3_train = hstack([E3_train, X3_train[num_cols3].to_numpy(dtype=np.float64)])
F3_test = hstack([E3_test, X3_test[num_cols3].to_numpy(dtype=np.float64)])
model3 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(F3_train, y3_train)
pred3 = model3.predict(F3_test)
accuracy3 = accuracy_score(y3_test, pred3)
print(f'Model 3 accuracy: {accuracy3:.2%}')
print(classification_report(y3_test, pred3))

## 9. Recorded random-split results

| Model | Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|
| Baseline | 63.84% | 0.541 | 0.658 |
| + Concentration | 66.04% | 0.543 | 0.672 |
| + Concentration + Temperature | **78.35%** | **0.701** | **0.789** |

Temperature produced the largest improvement in the random split.

## 10. Strict unseen-reference validation

Reference #253 contains 79.35% of all rated observations. To test cross-source generalization, it is kept completely out of training and used only as the test set.

In [ ]:
ref_df = rated_df[rated_df['Reference #'].notna()].copy()
Xr = ref_df[['material_group_clean','Material Family_clean','Material_clean','Environment_clean','concentration_mid','temperature_mid_c']]
yr = ref_df['corrosion_rating']
train_ref = ref_df['Reference #'] != 253
test_ref = ref_df['Reference #'] == 253
Xr_train = Xr[train_ref].copy(); Xr_test = Xr[test_ref].copy()
yr_train = yr[train_ref]; yr_test = yr[test_ref]
rc = Xr_train['concentration_mid'].median(); rt = Xr_train['temperature_mid_c'].median()
for c in cat:
    Xr_train[c] = Xr_train[c].fillna('Missing').astype(str)
    Xr_test[c] = Xr_test[c].fillna('Missing').astype(str)
Xr_train['concentration_missing'] = Xr_train['concentration_mid'].isna().astype(int); Xr_test['concentration_missing'] = Xr_test['concentration_mid'].isna().astype(int)
Xr_train['temperature_missing'] = Xr_train['temperature_mid_c'].isna().astype(int); Xr_test['temperature_missing'] = Xr_test['temperature_mid_c'].isna().astype(int)
Xr_train['concentration_mid'] = Xr_train['concentration_mid'].fillna(rc); Xr_test['concentration_mid'] = Xr_test['concentration_mid'].fillna(rc)
Xr_train['temperature_mid_c'] = Xr_train['temperature_mid_c'].fillna(rt); Xr_test['temperature_mid_c'] = Xr_test['temperature_mid_c'].fillna(rt)
encr = OneHotEncoder(handle_unknown='ignore')
Er_train = encr.fit_transform(Xr_train[cat]); Er_test = encr.transform(Xr_test[cat])
Fr_train = hstack([Er_train, Xr_train[num_cols3].to_numpy(dtype=np.float64)])
Fr_test = hstack([Er_test, Xr_test[num_cols3].to_numpy(dtype=np.float64)])
model_ref = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(Fr_train, yr_train)
pred_ref = model_ref.predict(Fr_test)
accuracy_ref = accuracy_score(yr_test, pred_ref)
print(f'Unseen Reference #253 accuracy: {accuracy_ref:.2%}')
print(classification_report(yr_test, pred_ref))

## 11. Final interpretation

The random split achieved 78.35% accuracy and 0.70 Macro F1, while the completely unseen Reference #253 holdout achieved 39.54% accuracy and 0.26 Macro F1.

The project therefore reports both results. The random-split score describes performance within the overall dataset distribution; it should not be presented as guaranteed performance on a previously unseen source document. The large gap demonstrates that validation strategy and source distribution matter substantially for this dataset.